# Lab 2 — Queries Avançadas e Investigação de Redes Criminosas
## Graph RAG para Análise Multi-hop com LightRAG

**Aula 9 | MBA: RAG & CAG Aplicados a Direito e Segurança Pública**

---

### Objetivo
Explorar o potencial do Graph RAG para:
- Mapeamento de redes criminosas via raciocínio multi-hop;
- Análise de jurisprudência com conexões cruzadas;
- Queries temáticas globais sobre corpus jurídico;
- Comparação quantitativa entre modos de query.

### Pré-requisito
Ter concluído o Lab 1 (LightRAG já configurado e corpus indexado).

## Passo 1: Reconectar ao LightRAG (continuação do Lab 1)

In [ ]:
# Passo 1.1 — Reinstalar dependências (se necessário)
!pip install "lightrag-hku[opensearch]" FlagEmbedding nest-asyncio --quiet

import asyncio
import nest_asyncio
import time
import os
nest_asyncio.apply()

from lightrag import LightRAG, QueryParam
from FlagEmbedding import FlagModel
import numpy as np

print("✅ Dependências carregadas")

In [ ]:
# Passo 1.2 — Recriar funções de embedding e LLM
# (mesma configuração do Lab 1 — copie suas variáveis de ambiente)

from lightrag.llm.openai import openai_complete_if_cache

VLLM_BASE_URL = os.getenv("VLLM_BASE_URL", "http://localhost:8000/v1")
VLLM_MODEL = os.getenv("VLLM_MODEL", "meta-llama/Llama-3.1-8B-Instruct")
VLLM_API_KEY = os.getenv("VLLM_API_KEY", "token-vllm")

# Modelo de embedding
embedding_model = FlagModel('BAAI/bge-m3', use_fp16=True)

async def embedding_func_bge(texts):
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(None, lambda: embedding_model.encode(texts, batch_size=16))

async def llm_model_func(prompt, system_prompt=None, history_messages=[], **kwargs):
    return await openai_complete_if_cache(
        model=VLLM_MODEL, prompt=prompt,
        system_prompt=system_prompt or "Você é um assistente jurídico especializado em direito penal brasileiro. Responda em português.",
        history_messages=history_messages,
        api_key=VLLM_API_KEY, base_url=VLLM_BASE_URL, **kwargs
    )

# Reconectar ao LightRAG (usa o working_dir existente com dados já indexados)
rag = LightRAG(
    llm_model_func=llm_model_func,
    embedding_func=embedding_func_bge,
    embedding_dim=1024,
    kv_storage="OpensearchKVStorage",
    vector_storage="OpensearchVectorDBStorage",
    graph_storage="OpensearchGraphStorage",
    doc_status_storage="JsonDocStatusStorage",
    working_dir="./lightrag_aula9",
    addon_params={
        "opensearch_config": {"host": "localhost", "port": 9200, "use_ssl": False, "verify_certs": False},
        "language": "Portuguese"
    }
)

print("✅ LightRAG reconectado ao índice existente")

## Passo 2: Queries de Investigação Criminal (Raciocínio Multi-hop)

In [ ]:
# Passo 2.1 — Cenário investigativo: mapeamento de atores e conexões
# Query multi-hop: Silva → criptomoedas → lavagem → organizações

async def query_investigativa(pergunta: str, modo: str = "hybrid", verbose: bool = True):
    """Executa query investigativa com contexto detalhado."""
    if verbose:
        print(f"\n🔍 INVESTIGAÇÃO: {pergunta}")
        print(f"   Modo: {modo}")
        print("-" * 60)
    
    inicio = time.time()
    resposta = await rag.aquery(pergunta, param=QueryParam(mode=modo))
    tempo = time.time() - inicio
    
    if verbose:
        print(resposta)
        print(f"\n⏱️ {tempo:.2f}s")
    
    return resposta

# Query 1: Cadeia de prova digital
q1 = "Quais são os requisitos legais para validade da prova digital em processos \
criminais e como a cadeia de custódia deve ser mantida?"

asyncio.run(query_investigativa(q1, modo="hybrid"))

In [ ]:
# Passo 2.2 — Query sobre hierarquia de organização criminosa
# Testa navegação pelo grafo de relações hierárquicas

q2 = "Como é estruturada hierarquicamente uma organização criminosa conforme \
identificado nas investigações e qual o papel legal de cada nível hierárquico?"

asyncio.run(query_investigativa(q2, modo="local"))

In [ ]:
# Passo 2.3 — Query sobre instrumentos de justiça negociada
# Testa navegação entre conceitos relacionados: ANPP, colaboração, requisitos

q3 = "Compare o Acordo de Não Persecução Penal (ANPP) com a Colaboração \
Premiada: requisitos, quem pode celebrar, benefícios e diferenças fundamentais."

asyncio.run(query_investigativa(q3, modo="hybrid"))

## Passo 3: Análise de Comunidades Temáticas (Modo Global)

In [ ]:
# Passo 3.1 — Queries globais para mapeamento temático do corpus

queries_globais = [
    "Quais são os principais temas jurídico-penais abordados no corpus?",
    "Quais instrumentos tecnológicos são mencionados como meios de prova ou crimes?",
    "Quais instituições do sistema de justiça criminal estão representadas e quais seus papéis?",
]

async def executar_queries_globais():
    for i, pergunta in enumerate(queries_globais, 1):
        print(f"\n{'='*60}")
        print(f"QUERY GLOBAL {i}: {pergunta}")
        print(f"{'='*60}")
        resposta = await rag.aquery(pergunta, param=QueryParam(mode="global"))
        print(resposta)

asyncio.run(executar_queries_globais())

## Passo 4: Benchmark Quantitativo — Comparação entre Modos

In [ ]:
# Passo 4.1 — Executar benchmark de tempo e qualidade

import pandas as pd

perguntas_benchmark = [
    {"id": "Q1", "tipo": "entidade_específica",
     "texto": "Quais são as penas previstas na Lei 12.850 para líderes de organizações criminosas?"},
    {"id": "Q2", "tipo": "multi_hop",
     "texto": "Como a colaboração premiada se relaciona com o ANPP e quais as diferenças de requisitos?"},
    {"id": "Q3", "tipo": "temático_global",
     "texto": "Quais são os padrões de uso de tecnologia em crimes financeiros identificados no corpus?"},
    {"id": "Q4", "tipo": "prova_digital",
     "texto": "Quais ferramentas forenses são recomendadas para investigação digital?"},
]

modos = ["naive", "local", "global", "hybrid"]
resultados_benchmark = []

async def executar_benchmark():
    for pergunta in perguntas_benchmark:
        for modo in modos:
            inicio = time.time()
            try:
                resposta = await rag.aquery(
                    pergunta["texto"],
                    param=QueryParam(mode=modo)
                )
                tempo = time.time() - inicio
                n_chars = len(resposta)
                
                resultados_benchmark.append({
                    "Query": pergunta["id"],
                    "Tipo": pergunta["tipo"],
                    "Modo": modo,
                    "Tempo (s)": round(tempo, 2),
                    "Tamanho Resp.": n_chars,
                })
                print(f"  ✅ {pergunta['id']} × {modo}: {tempo:.2f}s ({n_chars} chars)")
            except Exception as e:
                print(f"  ❌ {pergunta['id']} × {modo}: {e}")

print("🔄 Executando benchmark (pode demorar alguns minutos)...")
asyncio.run(executar_benchmark())

In [ ]:
# Passo 4.2 — Visualizar resultados do benchmark

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

# Se o benchmark não foi executado, usar dados simulados para demonstração
if not resultados_benchmark:
    print("ℹ️ Usando dados simulados para demonstração")
    dados_simulados = [
        # Q1 - entidade_específica
        {"Query": "Q1", "Tipo": "entidade_específica", "Modo": "naive", "Tempo (s)": 1.2, "Tamanho Resp.": 420},
        {"Query": "Q1", "Tipo": "entidade_específica", "Modo": "local", "Tempo (s)": 2.8, "Tamanho Resp.": 890},
        {"Query": "Q1", "Tipo": "entidade_específica", "Modo": "global", "Tempo (s)": 4.1, "Tamanho Resp.": 650},
        {"Query": "Q1", "Tipo": "entidade_específica", "Modo": "hybrid", "Tempo (s)": 5.3, "Tamanho Resp.": 1150},
        # Q2 - multi_hop
        {"Query": "Q2", "Tipo": "multi_hop", "Modo": "naive", "Tempo (s)": 1.4, "Tamanho Resp.": 380},
        {"Query": "Q2", "Tipo": "multi_hop", "Modo": "local", "Tempo (s)": 3.2, "Tamanho Resp.": 920},
        {"Query": "Q2", "Tipo": "multi_hop", "Modo": "global", "Tempo (s)": 4.5, "Tamanho Resp.": 780},
        {"Query": "Q2", "Tipo": "multi_hop", "Modo": "hybrid", "Tempo (s)": 6.1, "Tamanho Resp.": 1340},
        # Q3 - temático_global
        {"Query": "Q3", "Tipo": "temático_global", "Modo": "naive", "Tempo (s)": 1.1, "Tamanho Resp.": 290},
        {"Query": "Q3", "Tipo": "temático_global", "Modo": "local", "Tempo (s)": 2.9, "Tamanho Resp.": 670},
        {"Query": "Q3", "Tipo": "temático_global", "Modo": "global", "Tempo (s)": 3.8, "Tamanho Resp.": 1100},
        {"Query": "Q3", "Tipo": "temático_global", "Modo": "hybrid", "Tempo (s)": 5.5, "Tamanho Resp.": 1280},
        # Q4 - prova_digital
        {"Query": "Q4", "Tipo": "prova_digital", "Modo": "naive", "Tempo (s)": 1.3, "Tamanho Resp.": 450},
        {"Query": "Q4", "Tipo": "prova_digital", "Modo": "local", "Tempo (s)": 2.7, "Tamanho Resp.": 840},
        {"Query": "Q4", "Tipo": "prova_digital", "Modo": "global", "Tempo (s)": 3.9, "Tamanho Resp.": 720},
        {"Query": "Q4", "Tipo": "prova_digital", "Modo": "hybrid", "Tempo (s)": 5.0, "Tamanho Resp.": 1200},
    ]
    resultados_benchmark = dados_simulados

df = pd.DataFrame(resultados_benchmark)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
cores_modos = {"naive": "#FF6B6B", "local": "#4ECDC4", "global": "#45B7D1", "hybrid": "#96CEB4"}

# Gráfico 1: Tempo médio por modo
tempo_medio = df.groupby("Modo")["Tempo (s)"].mean()
ax1 = axes[0]
barras = ax1.bar(tempo_medio.index,
                  tempo_medio.values,
                  color=[cores_modos[m] for m in tempo_medio.index],
                  edgecolor='black', linewidth=0.5)
ax1.set_title("Tempo Médio de Resposta por Modo", fontweight='bold')
ax1.set_ylabel("Tempo (segundos)")
ax1.set_xlabel("Modo de Query")
for barra, val in zip(barras, tempo_medio.values):
    ax1.text(barra.get_x() + barra.get_width()/2., barra.get_height() + 0.05,
             f'{val:.1f}s', ha='center', va='bottom', fontweight='bold')

# Gráfico 2: Riqueza da resposta (tamanho) por modo
tamanho_medio = df.groupby("Modo")["Tamanho Resp."].mean()
ax2 = axes[1]
barras2 = ax2.bar(tamanho_medio.index,
                   tamanho_medio.values,
                   color=[cores_modos[m] for m in tamanho_medio.index],
                   edgecolor='black', linewidth=0.5)
ax2.set_title("Riqueza Média da Resposta por Modo", fontweight='bold')
ax2.set_ylabel("Caracteres na Resposta")
ax2.set_xlabel("Modo de Query")
for barra, val in zip(barras2, tamanho_medio.values):
    ax2.text(barra.get_x() + barra.get_width()/2., barra.get_height() + 10,
             f'{int(val)}', ha='center', va='bottom', fontweight='bold')

plt.suptitle("Benchmark LightRAG — Corpus Jurídico Brasileiro",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('benchmark_modos.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Tabela de Resultados:")
print(df.to_string(index=False))

## Passo 5: Busca Direta no Índice OpenSearch (Bypass do RAG)

In [ ]:
# Passo 5.1 — Busca direta de entidades no OpenSearch
# Útil para inspeção forense do grafo e debug

import requests
import json

OPENSEARCH_URL = "http://localhost:9200"

def buscar_entidade_opensearch(nome_entidade: str, index: str = "aula9-entities"):
    """Busca uma entidade diretamente no índice OpenSearch."""
    query = {
        "query": {
            "match": {
                "entity_name": nome_entidade
            }
        },
        "size": 5
    }
    
    try:
        resp = requests.post(
            f"{OPENSEARCH_URL}/{index}/_search",
            json=query,
            headers={"Content-Type": "application/json"}
        )
        
        if resp.status_code == 200:
            dados = resp.json()
            hits = dados['hits']['hits']
            print(f"\n🔍 Entidade '{nome_entidade}': {len(hits)} resultado(s)")
            for hit in hits:
                source = hit['_source']
                print(f"\n  ID: {hit['_id']}")
                print(f"  Nome: {source.get('entity_name', 'N/A')}")
                print(f"  Tipo: {source.get('entity_type', 'N/A')}")
                desc = source.get('description', '')[:200]
                print(f"  Descrição: {desc}...")
        else:
            print(f"Status {resp.status_code}: {resp.text[:200]}")
    except Exception as e:
        print(f"Erro: {e}")

# Buscar entidades-chave do corpus
for entidade in ["Colaboração_Premiada", "Lei_12850", "Organização_Criminosa"]:
    buscar_entidade_opensearch(entidade)

In [ ]:
# Passo 5.2 — Busca vetorial kNN diretamente no OpenSearch
# Permite busca semântica sem passar pelo pipeline do LightRAG

async def busca_vetorial_direta(texto_query: str, top_k: int = 5):
    """
    Realiza busca semântica diretamente no índice vetorial do OpenSearch.
    Útil para verificar quais entidades são semanticamente mais próximas de uma query.
    """
    # Gerar embedding da query
    vetor_query = await embedding_func_bge([texto_query])
    vetor_lista = vetor_query[0].tolist()
    
    # Query kNN no OpenSearch
    query_knn = {
        "query": {
            "knn": {
                "embedding": {
                    "vector": vetor_lista,
                    "k": top_k
                }
            }
        },
        "_source": ["entity_name", "entity_type", "description"],
        "size": top_k
    }
    
    try:
        resp = requests.post(
            f"{OPENSEARCH_URL}/aula9-entities/_search",
            json=query_knn,
            headers={"Content-Type": "application/json"}
        )
        
        if resp.status_code == 200:
            dados = resp.json()
            hits = dados['hits']['hits']
            print(f"\n🔎 Busca semântica: '{texto_query}'")
            print(f"Top-{top_k} entidades mais relevantes:")
            for i, hit in enumerate(hits, 1):
                s = hit['_source']
                score = hit['_score']
                print(f"  [{i}] {s.get('entity_name')} ({s.get('entity_type')}) — score: {score:.4f}")
        else:
            print(f"Status {resp.status_code}")
    except Exception as e:
        print(f"Erro na busca vetorial: {e}")

asyncio.run(busca_vetorial_direta("instrumentos jurídicos para delação"))
asyncio.run(busca_vetorial_direta("rastreamento de ativos digitais em crimes financeiros"))

## Passo 6: Exercício Avaliativo

### Cenário: Você é analista de inteligência policial

Foi descoberto que o réu Rodrigo Menezes (identificado na Operação Hydra, inserida no Lab 1) possui conexões com o caso de criptomoedas da Operação Integração e também com o precedente jurídico do HC 127.483 do STF.

**Tarefa:** usando o LightRAG, construa um relatório investigativo respondendo às perguntas abaixo.

In [ ]:
# Exercício 1: Identifique as conexões entre as operações
# Dica: use modo "local" para entidades específicas

# ESCREVA SUA QUERY AQUI:
pergunta_ex1 = """[Substitua por sua pergunta sobre as conexões entre 
                  Operação Hydra, Operação Integração e lavagem via criptomoedas]"""

# asyncio.run(query_investigativa(pergunta_ex1, modo="local"))

In [ ]:
# Exercício 2: Identifique quais instrumentos jurídicos podem ser aplicados
# Dica: use modo "hybrid" para conectar entidades e temas

# ESCREVA SUA QUERY AQUI:
pergunta_ex2 = """[Substitua por sua pergunta sobre quais meios de obtenção de prova 
                  e instrumentos de justiça negociada são aplicáveis ao caso]"""

# asyncio.run(query_investigativa(pergunta_ex2, modo="hybrid"))

In [ ]:
# Exercício 3: Análise panorâmica do corpus para embasar relatório
# Dica: use modo "global" para visão temática

# ESCREVA SUA QUERY AQUI:
pergunta_ex3 = """[Substitua por sua pergunta sobre os padrões gerais de 
                  crimes financeiros e tecnológicos identificados no corpus]"""

# asyncio.run(query_investigativa(pergunta_ex3, modo="global"))

## Resumo do Lab 2

Neste laboratório você:

1. ✅ Executou queries investigativas de raciocínio multi-hop
2. ✅ Realizou análise temática global do corpus jurídico
3. ✅ Comparou quantitativamente tempo e riqueza de resposta entre os 4 modos
4. ✅ Acessou diretamente o índice OpenSearch para inspeção forense
5. ✅ Executou busca vetorial kNN diretamente no OpenSearch

**Conclusão prática:** O modo **hybrid** oferece a melhor qualidade para investigações criminais complexas, enquanto o modo **local** é mais eficiente para perguntas sobre entidades específicas. O modo **global** é indispensável para mapeamento temático de grandes acervos.

**Próximo Lab:** Lab 3 — Pipeline Jurídico Completo: Graph RAG + Busca Híbrida + Reranking